In [ ]:
import cv2
import random
import logging
import requests  # type: ignore
import sys
import time

from jetbot import Camera, Robot  # type: ignore


def setup_logging() -> logging.Logger:
    """Setup client logging"""
    lg = logging.getLogger("app_logger")

    if not lg.handlers:
        logging.basicConfig(level=logging.DEBUG)
        log_format = logging.Formatter("%(levelname)s - %(message)s")
        stream_handler = logging.StreamHandler(sys.stdout)
        stream_handler.setFormatter(log_format)
        file_handler = logging.FileHandler(f"app_{BOT_COLOR}.log")
        file_handler.setFormatter(log_format)

        lg = logging.getLogger("app_logger")
        lg.addHandler(file_handler)
        lg.addHandler(stream_handler)
        lg.propagate = False

    return lg


# --- CONFIGURATION ---
BOT_COLOR = "green"  # set bot color here
SERVER_BASE_URL = "http://194.47.156.103:5000"  # set server ip adress here
SERVER_URL = f"{SERVER_BASE_URL}/infer/{BOT_COLOR}"

STOPPING_AREA = 80000
STOP_AREA_BOT = 60000
MAX_SPEED = 0.4

is_stuck = False

# setup logging
lg = setup_logging()
logging.getLogger("Adafruit_I2C").setLevel(logging.WARNING)
logging.getLogger("Adafruit_MotorHAT").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

# initialize camera and robot
camera = Camera.instance(width=640, height=640)
robot = Robot()


def main():
    """Main function"""
    global is_stuck
    lg.info(f"Starting {BOT_COLOR} - Waiting for Server Commands at {SERVER_URL}...")

    # define variables
    prev_time = time.time()
    Kp = 0.5
    base_speed = 0.2
    PULSE_TIME = 0.11

    max_rotate_time = 13
    rotate_time_start = 0
    searching = False
    direction = 1

    # Stuck detection state
    tracked_label = None
    best_area = 0
    last_progress_time = time.time()
    last_found = time.time()
    PROGRESS_AREA_DELTA = 70
    STUCK_TIMEOUT = 7.0

    # Main loop for running
    while True:
        # get camera frame
        frame = camera.value
        _, img_encoded = cv2.imencode(".jpg", frame)

        try:
            # send image to server, wait for response
            response = requests.post(
                SERVER_URL,
                files={"image": img_encoded.tobytes()},
                timeout=1.5,
            )
            request_end_time = time.time()

            # find response time
            dt = request_end_time - prev_time
            if dt <= 0:
                dt = 0.01

            data = response.json()
            status = data.get("status")

            # if status is found we have a target to drive towards
            if status == "found":
                searching = False
                last_found = time.time()

                label = data.get("label")
                error = data.get("steering_bias")
                area = data.get("box_area") or 0

                print(
                    f"Tracking {label} | Error: {error} | Area: {area} | Latency: {dt:.3f}s"
                )

                # Reset stuck tracking when switching between box / bot / another target
                if label != tracked_label:
                    tracked_label = label
                    best_area = area
                    last_progress_time = time.time()
                    is_stuck = False

                # Detect forward progress by checking whether target area is increasing
                if area > best_area + PROGRESS_AREA_DELTA:
                    best_area = area
                    print("Progress", area, best_area)
                    last_progress_time = time.time()
                    is_stuck = False

                time_since_progress = time.time() - last_progress_time

                # Stop conditions
                if label == "box" and area > STOPPING_AREA:
                    print("Box reached!")
                    robot.stop()
                    is_stuck = False
                    continue

                elif label != "box" and label is not None and area > STOP_AREA_BOT:
                    print("Bot reached!")
                    robot.stop()
                    is_stuck = False
                    continue

                # Stuck condition for both box and bots
                elif label is not None and time_since_progress > STUCK_TIMEOUT:
                    print(f"Stuck while tracking {label}. Backing up and trying again.")
                    is_stuck = True
                    robot.backward(0.1)
                    time.sleep(3)

                    if random.random() < 0.5:
                        robot.left(0.1)
                    else:
                        robot.right(0.1)

                    time.sleep(3)
                    robot.stop()

                    # Reset progress tracking after recovery movement
                    best_area = area
                    last_progress_time = time.time()
                    prev_time = request_end_time

                    is_stuck = False
                    continue

                # Drive towards target
                else:
                    if error is None:
                        robot.stop()
                        continue

                    if abs(error) < 0.05:
                        turn = 0.0
                    else:
                        turn = Kp * error

                    # calculate motor speeds
                    left_motor = max(0.0, min(MAX_SPEED, base_speed + turn))
                    right_motor = max(0.0, min(MAX_SPEED, base_speed - turn))

                    # turn for pulse time duration
                    robot.set_motors(left_motor, right_motor)
                    time.sleep(PULSE_TIME)

                    # drive straight with base speed
                    robot.set_motors(base_speed, base_speed)
                    prev_time = request_end_time

            elif status == "searching":
                # Search for targets
                if time.time() - last_found < 3:
                    robot.set_motors(0.1, 0.1)
                if not searching:
                    rotate_time_start = time.time()
                    direction = random.choice([1, -1])

                lg.info("No targets found. Wandering/Searching...")

                # if robot has rotated for a certain time, drive forward or backwards instead
                if time.time() - rotate_time_start > max_rotate_time:
                    robot.set_motors(0.1 * direction, 0.1 * direction)

                    if time.time() - rotate_time_start > 2 + max_rotate_time:
                        # reset and go back to rotating
                        rotate_time_start = time.time()
                        direction = random.choice([1, -1])
                else:
                    robot.left(0.1 * direction)

                prev_time = request_end_time
                searching = True

                # Reset stuck detection when target is lost
                tracked_label = None
                best_area = 0
                last_progress_time = time.time()
                is_stuck = False

            else:
                robot.stop()

            if not searching:
                rotate_time_start = 0

        except requests.exceptions.RequestException as e:
            print(f"Network error: Cannot reach server. {e}")
            robot.stop()
            time.sleep(1)

        time.sleep(0.05)


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("Shutting down...")
        robot.stop()
